# Choptyuk a-C ↔ QCD θ: numerical notebook

Companion to **`choptyuk_qcd_bridge.pdf`** in `docs/monograph/qcd_bridge/`.

This notebook reproduces all five numerical experiments described in the monograph and lets you interactively explore variations.

**Five experiments:**

| ID | Question |
|----|----------|
| E1 | Which SU(3)-analogues of $\delta_C=\pi/7$ are candidates for the Choptyuk formula $\delta^5/b=10^{-10}$? |
| E2 | What are the Betti numbers of SU(N) instanton moduli spaces? |
| E3 | Scale-bridge: $a_C \cdot (\Lambda_{\mathrm{QCD}}/M_X)^p = 10^{-10}$ — for which $M_X$, $p$? |
| E4 | Power-law decompositions of θ — which $(\delta, b, n)$ yield $10^{-10}$? |
| E5 | Broad $(\delta, b)$ sweep — visualization of $\log_{10}(\delta^5/b)$. |

**Headline findings:**
- $\delta = \pi/168$ (full PSL(2,7) order) with $b=22$: $\delta^5/b \approx 1.04\times 10^{-10}$ — within 4% of θ bound
- $a_C \cdot (\Lambda_{\mathrm{QCD}}/M_{\mathrm{Pl}})^{1/3} \approx 2.1\times 10^{-10}$ — within a factor of 2
- Direct identification refuted; bridge hypothesis remains numerological but not refuted at the level of structural possibility.

## 0. Setup

In [ ]:
import math, json, os
from dataclasses import dataclass, asdict
from typing import List, Dict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']
mpl.rcParams['axes.unicode_minus'] = False
mpl.rcParams['figure.dpi'] = 110

# Constants
THETA_TARGET = 1e-10
A_C = (math.pi / 7) ** 5 / 22
LAMBDA_QCD_MEV = 200.0

MASS_SCALES_GEV = {
    'Planck': 1.22e19,
    'GUT':    1.0e16,
    'SUSY':   1.0e3,
    'top':    173.0,
    'W':      80.4,
    'Higgs':  125.0,
}

print(f'a_C (Choptyuk) = {A_C:.6e}')
print(f'θ_QCD target   = {THETA_TARGET:.0e}')
print(f'log10(a_C)     = {math.log10(A_C):+.4f}')
print(f'log10(target)  = {math.log10(THETA_TARGET):+.4f}')
print(f'orders of mag. = {math.log10(A_C) - math.log10(THETA_TARGET):.2f}')

## E1. SU(3)-analogues of $\delta_C = \pi/7$

We test all mathematically motivated candidates for the QCD analogue of $\delta_C$, with several Betti-number candidates $b$.

In [ ]:
PI = math.pi

@dataclass
class DeltaCandidate:
    name: str
    value: float
    source: str

candidates = [
    DeltaCandidate('pi/N (SU(3))',         PI/3,   'Coxeter number of SU(N)=N'),
    DeltaCandidate('pi/h^∨ (SU(3))',        PI/3,   'dual Coxeter h^∨=N for SU(N)'),
    DeltaCandidate('pi/2N (SU(3))',         PI/6,   'center Z_N smallest generator'),
    DeltaCandidate('2pi/N (center, SU(3))', 2*PI/3, 'center Z_N element'),
    DeltaCandidate('Cartan long root',      PI/3,   'long-root Weyl reflection'),
    DeltaCandidate('Cartan short (SU(3))',  2*PI/3, 'short-root (adjoint weight)'),
    DeltaCandidate('pi/7 (Klein, Choptyuk)', PI/7,  'Hurwitz Γ(2,3,7) order-7 generator'),
    DeltaCandidate('pi/|PSL(2,7)|=pi/168',  PI/168, 'order of PSL(2,7) full group'),
    DeltaCandidate('pi/dim(SU(3))=pi/8',    PI/8,   'dimension of adjoint rep'),
    DeltaCandidate('pi/(N+1) affine',       PI/4,   'affine Coxeter h=N+1'),
    DeltaCandidate('pi/N^2 (SU(3))',        PI/9,   'higher instanton winding'),
    DeltaCandidate('pi/N^3 (SU(3))',        PI/27,  'third-order winding'),
    DeltaCandidate('pi/(N·h^∨)=pi/9',       PI/9,   'rank × Coxeter'),
    DeltaCandidate('pi/100',                PI/100, 'ad-hoc scale-suppressed'),
    DeltaCandidate('pi/200',                PI/200, '(Λ_QCD/E_Planck)^{-1/2} ∼ 1/200'),
]

b_candidates = [
    ('b2=22 (K3)',               22),
    ('b2=2 (Â(K3) Dirac index)', 2),
    ('b2=k=1 (SU(N) inst.)',     1),
    ('b2=k=2 (SU(2), charge 2)', 2),
    ('b2=k=3 (SU(2), charge 3)', 3),
    ('b2=dim(SU(3))=8',          8),
    ('b2=22·8=176 (K3×SU(3)?)',  176),
    ('b2=24 (Leech lattice)',    24),
    ('b2=12 (SU(3) charge-1)',   12),
]

rows = []
for c in candidates:
    for name_b, b in b_candidates:
        v = c.value ** 5 / b
        log_v = math.log10(v) if v > 0 else -100
        rows.append({**asdict(c), 'b_name': name_b, 'b': b, 'value': v, 'log10': log_v,
                     'distance': abs(log_v + 10)})

rows.sort(key=lambda r: r['distance'])
print(f'{"#":>3} {"δ candidate":32s} {"b name":30s} {"log10(δ^5/b)":>15s} {"dist":>8s}')
print('-' * 90)
for i, r in enumerate(rows[:15]):
    print(f'{i+1:>3} {r["name"]:32s} {r["b_name"]:30s} {r["log10"]:>+15.4f} {r["distance"]:>8.4f}')

### ⭐ E1 Best match: $\delta = \pi/168$ with $b=22$

$(\pi/168)^5 / 22 \approx 1.04 \times 10^{-10}$ — within 4% of θ bound. The angle $\pi/168$ corresponds to the **full automorphism group order** of PSL(2,7), rather than a single generator.

Caveat: holonomies are typically $\exp(2\pi i/n)$ for generators of order $n$, not for whole groups. Treating $\pi/168$ as a holonomy angle is geometrically unusual.

In [ ]:
best = rows[0]
print(f'Best match: δ = {best["name"]} = {best["value"]:.6e}')
print(f'  with b = {best["b_name"]} (b={best["b"]})')
print(f'  δ^5/b = {best["value"]**5 / best["b"]:.4e}  (target: 1e-10)')
print(f'  log10 = {best["log10"]:+.4f}  (target: -10.000)')
print(f'  distance from target: {best["distance"]:.4f} log10 units')

## E2. Betti numbers of SU(N) instanton moduli spaces

The natural QCD-side denominator for the Choptyuk formula is the Betti number of the instanton moduli space $M_{k,N}$ (charge-$k$ instantons on SU(N)).

Universal result from ADHM: $b_2(M_{k,N}) = k$ for all $N \geq 2$, $k \geq 1$.

In [ ]:
print(f'{"Group":>8} {"k":>3} {"dim_R":>6} {"b_2":>4} {"χ (approx)":>12}')
print('-' * 40)
for N in [2, 3, 4, 5]:
    for k in [1, 2, 3, 4, 5]:
        dim = 4 * N * k
        b2 = k
        chi = math.comb(N + k - 1, k)
        print(f'SU({N})   {k:>3} {dim:>6} {b2:>4} {chi:>12}')

print()
print('For comparison:')
print(f'  K3 surface: dim=4, b_2=22, χ=24')
print(f'  Hilb^k(K3): b_2=23 for all k ≥ 1 (Beauville)')

## E3. Scale-bridge: $a_C \cdot (\Lambda_{\mathrm{QCD}}/M_X)^p = 10^{-10}$

Test the user's hypothesis: $a_C$ acts at particle/QNM scale and must be rescaled to reach QCD scale. Find $p$ such that $a_C \cdot (\Lambda_{\mathrm{QCD}}/M_X)^p = 10^{-10}$.

In [ ]:
ratio = THETA_TARGET / A_C
log10_ratio = math.log10(ratio)
print(f'a_C = {A_C:.6e}, target = {THETA_TARGET:.0e}, target/a_C = {ratio:.4e}')
print(f'log10(target/a_C) = {log10_ratio:+.4f}')
print()
print(f'{"M_X":>10} {"M_X [GeV]":>14} {"Λ_QCD/M_X":>14} {"p needed":>10} {"nearest nice":>15}')
print('-' * 70)

for name, M_gev in MASS_SCALES_GEV.items():
    r = (LAMBDA_QCD_MEV / 1000) / M_gev
    log_r = math.log10(r)
    p = log10_ratio / log_r
    # Find nearest nice rational
    nice = None
    for q in [1, 2, 3, 4, 5, 6, 7, 8, 10]:
        for n in range(1, 12):
            if abs(p - n/q) < 0.02:
                nice = f'{n}/{q}'
                break
        if nice: break
    print(f'{name:>10} {M_gev:>14.3e} {r:>14.3e} {p:>10.4f} {(nice or "—"):>15}')

### ⭐ E3 Best match: Planck scale with $p \approx 1/3$

$a_C \cdot (\Lambda_{\mathrm{QCD}}/M_{\mathrm{Pl}})^{1/3} \approx 2.1 \times 10^{-10}$

The $1/3$ exponent is suggestive but unexplained. Possible physical interpretations:
- One-loop anomalous dimension of a dimension-3 operator
- Three colours of QCD (cube root of an SU(3) Casimir)
- Three generations of fermions
- Dimensional reduction $4D \to 1D$

None is a derivation.

In [ ]:
# Verify the Planck-scale 1/3 match
ratio_planck = LAMBDA_QCD_MEV / (MASS_SCALES_GEV['Planck'] * 1000)
value_planck = A_C * ratio_planck ** (1/3)
print(f'a_C × (Λ_QCD/M_Pl)^(1/3) = {value_planck:.4e}')
print(f'log10 = {math.log10(value_planck):+.4f}  (target = -10.000)')
print(f'distance from target: {abs(math.log10(value_planck) + 10):.4f} log10 units')
print(f'ratio to target: {value_planck / THETA_TARGET:.2f}×')

## E4. Power-law decompositions

Test the generalized Choptyuk formula $\delta^n / b = 10^{-10}$ for various $n$ and $b$. With $\delta = \pi/7$ fixed, what power $n$ reproduces the target?

In [ ]:
pi_over_7 = PI / 7
print('Searching for (π/7)^n / b ≈ 1e-10 ...')
print(f'{"n":>4} {"b":>5} {"log10(value)":>14} {"dist":>8}')
print('-' * 35)
matches = []
for n in range(2, 35):
    for b in [1, 2, 8, 12, 22, 23, 24, 176]:
        v = pi_over_7 ** n / b
        log_v = math.log10(v) if v > 0 else -100
        d = abs(log_v + 10)
        if d < 1.5:
            matches.append((n, b, log_v, d))
matches.sort(key=lambda x: x[3])
for n, b, lv, d in matches[:10]:
    print(f'{n:>4} {b:>5} {lv:>+14.4f} {d:>8.4f}')

print()
print('Conclusion: π/7 needs n ≈ 25 to reach 1e-10 — unnatural.')

## E5. Broad $(\delta, b)$ sweep — visualization

Plot the Choptyuk surface $\log_{10}(\delta^5/b)$ and identify the contour at $\log_{10} = -10$ (the $\theta_{\mathrm{QCD}}$ target).

In [ ]:
delta_grid = np.logspace(-3, 0, 400)
b_grid = np.linspace(1, 200, 400)
D, B = np.meshgrid(delta_grid, b_grid)
Z = np.log10(D ** 5 / B)

fig, ax = plt.subplots(figsize=(10, 7), constrained_layout=True)
levels = np.arange(-15, 1.5, 0.5)
cs = ax.contourf(D, B, Z, levels=levels, cmap='viridis_r', extend='both')
fig.colorbar(cs, ax=ax, label=r'$\log_{10}(\delta^5 / b)$')
cs2 = ax.contour(D, B, Z, levels=[-10], colors='red', linewidths=2.5)
ax.clabel(cs2, fmt=r'$\log_{10}=-10$ ($\theta_{\mathrm{QCD}}$)', fontsize=10, colors='red')

# Mark the a_C point
ax.plot(PI/7, 22, 'o', color='#FF6B35', markersize=14,
        markeredgecolor='white', markeredgewidth=2,
        label=r'$a_C$ Choptyuk: $\delta=\pi/7$, $b_2=22$')
ax.annotate(f'$a_C$ = {A_C:.2e}\nlog10 = -3.08',
            xy=(PI/7, 22), xytext=(PI/7 * 4, 50),
            fontsize=10, color='#FF6B35',
            arrowprops=dict(arrowstyle='->', color='#FF6B35', lw=1.5))

# Mark SU(3) candidates
for d, b, lab in [(PI/3, 1, r'$\pi/3$, $b=1$'), (PI/6, 1, r'$\pi/6$, $b=1$'),
                   (PI/9, 1, r'$\pi/9$, $b=1$'), (PI/168, 22, r'$\pi/168$, $b=22$')]:
    ax.plot(d, b, 's', color='#00C2A8', markersize=10,
            markeredgecolor='white', markeredgewidth=1.5)
    ax.annotate(lab, xy=(d, b), xytext=(d * 1.4, b + 12), fontsize=9, color='#006B6B')

ax.set_xscale('log')
ax.set_xlabel(r'$\delta$ (holonomy angle)', fontsize=12)
ax.set_ylabel(r'$b$ (Betti-like denominator)', fontsize=12)
ax.set_title(r'Choptyuk surface $\log_{10}(\delta^5/b)$ — red contour = $\theta_{\mathrm{QCD}}$ target', fontsize=12)
ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
plt.show()

## Verdict

The direct identification of $a_C$ with $\theta_{\mathrm{QCD}}$ is **refuted**:
- Magnitudes differ by ~7 orders of magnitude
- CP properties differ (CP-even vs CP-odd)
- Continuity differs (discrete vs continuous)
- Dynamical role differs (fixed vs relaxable via PQ)

But three numerical coincidences warrant further scrutiny:
1. $\delta = \pi/168$ (full PSL(2,7) order), $b = 22$: $\delta^5/b \approx 1.04 \times 10^{-10}$
2. $a_C \cdot (\Lambda_{\mathrm{QCD}}/M_{\mathrm{Pl}})^{1/3} \approx 2.1 \times 10^{-10}$
3. $a_C \cdot (\Lambda_{\mathrm{QCD}}/M_H)^{5/2} \approx 8.5 \times 10^{-11}$

Each requires an unexplained parameter (group-order promotion, $1/3$ exponent, $5/2$ exponent). Without derivations, the bridge hypothesis remains **numerological but not refuted at the structural level**.

**Recommended next steps:**
1. Derive the $1/3$ exponent from a one-loop anomalous dimension calculation
2. Search for a geometric realisation of $\pi/168$ as a holonomy angle
3. Compare bridge predictions against lattice QCD data on $\theta$-dependence

See `choptyuk_qcd_bridge.pdf` for the full theoretical discussion.